<a href="https://colab.research.google.com/github/PercyTechX/del-excel-al-ml/blob/main/temporada-1/08-cuando-el-85-no-sirve/solucion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Solución 1.8 · Cuando el 85 % de aciertos no sirve

**Serie:** Del Excel al Machine Learning · **Temporada 1** · 🎬 Video próximamente

Estas son las respuestas del ejercicio. Si la tuya es distinta pero da el mismo resultado, ¡también está bien! En programación suele haber más de una forma de llegar a la respuesta.

### Prepara los datos

Ejecuta estas cuatro celdas. Dejan listos `real`, `probabilidad` y la función `costo_total`.

Algunos ejercicios usan variables de los anteriores, así que **ejecútalos en orden**.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATOS = "https://raw.githubusercontent.com/PercyTechX/del-excel-al-ml/main/datos/"

In [2]:
def preparar_x(tabla, numeros, textos, columnas=None):
    """Convierte los textos en columnas de 0 y 1. Con `columnas`, usa las de entrenamiento."""
    x = pd.get_dummies(tabla[numeros + textos], columns=textos, dtype=float)
    if columnas is not None:
        x = x.reindex(columns=columnas, fill_value=0)
    return x

In [3]:
def preparar_creditos(creditos):
    """Agrega las variables del modelo y separa entrenamiento y prueba al 1 de julio de 2025."""
    tabla = creditos.assign(
        log_monto=np.log(creditos["monto_sobre_limite"]),
        tasa_previa=(creditos["morosos_previos"] / creditos["casos_previos"]).fillna(0),
        sin_historial=(creditos["casos_previos"] == 0).astype(int),
    )
    fecha = pd.to_datetime(tabla["fecha"])
    se_sabe = fecha + pd.to_timedelta(tabla["plazo_dias"] + 31, unit="D")
    con_resultado = tabla["moroso"].notna()
    corte = pd.Timestamp("2025-07-01")
    return tabla[(se_sabe <= corte) & con_resultado], tabla[(fecha >= corte) & con_resultado]

In [4]:
creditos = pd.read_csv(DATOS + "creditos.csv")

entreno, prueba = preparar_creditos(creditos)

numeros = ["log_monto", "antiguedad_anios", "tasa_previa", "sin_historial"]
textos = ["tipo"]

x_entreno = preparar_x(entreno, numeros, textos)
x_prueba = preparar_x(prueba, numeros, textos, x_entreno.columns)

modelo = make_pipeline(StandardScaler(), LogisticRegression())
modelo.fit(x_entreno, entreno["moroso"])

probabilidad = modelo.predict_proba(x_prueba)[:, 1]

real = prueba["moroso"].astype(int).to_numpy()
tasa_previa = prueba["tasa_previa"].to_numpy()


def costo_total(real, probabilidad, umbral, costo_revisar, costo_escape):
    marcadas = probabilidad >= umbral
    revisadas = marcadas.sum()
    escapadas = (real == 1).sum() - real[marcadas].sum()
    return revisadas * costo_revisar + escapadas * costo_escape

### 1. Un detector inútil

Crea `nadie = probabilidad >= 2`, que no marca a nadie. Calcula su exactitud y su exhaustividad. Deben dar cerca de `0.854` y `0`. ¿Qué te dice eso de la exactitud? Responde en un comentario.

> 💡 No calcules su precisión: como no hay nada marcado, no se puede calcular y daría una advertencia.

In [5]:
nadie = probabilidad >= 2

print(round(accuracy_score(real, nadie), 3))
print(recall_score(real, nadie))

# Acierta mucho sin detectar nada: cuando un caso es raro, la exactitud no dice casi nada.

0.854
0.0


### 2. Otro umbral, a mano

Con un umbral de `0.2`, arma la matriz de confusión. Debes ver `116` morosas encontradas y `289` falsas alarmas. Con esas cifras, calcula **a mano** la precisión y la exhaustividad, y compruébalas con `precision_score` y `recall_score`.

La mora total es `223`.

In [6]:
marcadas = probabilidad >= 0.2

print(pd.crosstab(real, marcadas, rownames=["Real"], colnames=["Marcada"]))

print(round(116 / (116 + 289), 3), round(precision_score(real, marcadas), 3))
print(round(116 / 223, 3), round(recall_score(real, marcadas), 3))

Marcada  False  True 
Real                 
0         1019    289
1          107    116
0.286 0.286
0.52 0.52


### 3. Marcarlos a todos

Crea `todos = probabilidad >= 0`. ¿Cuál es su exhaustividad? ¿Y su precisión? ¿Sirve? Responde en un comentario.

In [7]:
todos = probabilidad >= 0

print(recall_score(real, todos))
print(round(precision_score(real, todos), 3))

# Encuentra toda la mora, pero revisa las 1,531 compras. Su precisión es solo la tasa de mora.

1.0
0.146


### 4. Cambia los costos

Usa `costo_total` para probar los umbrales `0`, `0.1`, `0.15`, `0.2`, `0.25`, `0.3`, `0.4`, `0.5` y `2` con estos costos, y di cuál es el mejor en cada caso. Recuerda: `0` revisa todas, y `2` no revisa ninguna.

- a) Revisar cuesta S/ 30 y un escape, S/ 60. El mejor debe ser `2`.
- b) Revisar cuesta S/ 10 y un escape, S/ 200. El mejor debe ser `0`.

¿Por qué cambia? Responde en un comentario.

In [8]:
umbrales = [0, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 2]

for umbral in umbrales:
    print(
        umbral,
        costo_total(real, probabilidad, umbral, 30, 60),
        costo_total(real, probabilidad, umbral, 10, 200),
    )

# Si revisar es caro, no conviene revisar a nadie. Si dejar escapar es caro, conviene revisar todas.
# En los dos extremos, el modelo no ayuda: solo sirve cuando los costos están en un punto medio.

0 45930 15310
0.1 37770 16370
0.15 26640 19500
0.2 18570 25450
0.25 15090 29870
0.3 13860 35040
0.4 13740 42380
0.5 13440 44260
2 13380 44600


### 5. El informe

Tu gerente lee: *"Nuestro modelo tiene 85 % de exactitud"*. Escribe en un comentario, en 3 líneas, qué le responderías, con cifras de este ejercicio.

In [9]:
# Decir que nadie cae en mora ya acierta el 85.4 %, así que ese 85 % no prueba nada.
# Lo que importa es cuánta mora encuentra y cuántas falsas alarmas da:
# con umbral 0.2, encuentra el 52 %, y acierta en el 29 % de lo que marca.

## 🎉 ¡Bien hecho!

Nos vemos en el **episodio 1.9: Miniproyecto, el pronóstico de la cartera**.

---

*Del Excel al Machine Learning* · por [Percy Nazario](https://www.linkedin.com/in/percy-nazario/) · [Repositorio](https://github.com/PercyTechX/del-excel-al-ml) · Contenido bajo licencia CC BY 4.0